### Analyse corrigée — EDSC Cameroun 2018
**Encodage correct des variables catégorielles + Variables ajoutées + Tests de validation complets**


In [19]:
import pandas as pd
import numpy as np
import pyreadstat
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit
from scipy.stats import chi2 as chi2_dist

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, learning_curve)
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from xgboost                 import XGBClassifier
from imblearn.over_sampling  import SMOTE
from sklearn.calibration     import calibration_curve
from sklearn.metrics         import (roc_auc_score, roc_curve, confusion_matrix,
                                      brier_score_loss, accuracy_score,
                                      f1_score, precision_score, recall_score)

os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables',  exist_ok=True)

plt.rcParams['figure.dpi']        = 130
plt.rcParams['font.family']       = 'DejaVu Sans'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid')

NAVY   = '#0D1B4B'
BLUE   = '#2196F3'
TEAL   = '#0097A7'
GREEN  = '#388E3C'
CORAL  = '#FF5722'
AMBER  = '#F57C00'
GRAY   = '#90A4AE'
PURPLE = '#7B1FA2'
COLORS_6 = [NAVY, TEAL, CORAL, AMBER, GREEN, BLUE]

print('✓ Bibliothèques chargées')

✓ Bibliothèques chargées


In [20]:

# ÉTAPE 1 — Encodage correct de toutes les variables

print('Chargement de CMIR71FL.SAV...')
df_raw, meta = pyreadstat.read_sav('../data/CMIR71FL.SAV')
df_raw.columns = df_raw.columns.str.lower()
print(f'  Données brutes : {df_raw.shape[0]:,} femmes · {df_raw.shape[1]} variables')

df = df_raw.copy()

#  Quantitatives (inchangées) 
df['age']        = df['v012'].astype(float)
df['nb_enfants'] = df['v218'].astype(float)

#  Ordinales (effet linéaire supposé) 
df['instruction'] = df['v106'].astype(float)   # 0=Aucun 1=Prim 2=Sec 3=Sup
df['quintile']    = df['v190'].astype(float)   # 1=Très pauvre → 5=Très riche

#  Binaires 
df['emploi']          = df['v714'].astype(float)
df['residence_rural'] = (df['v025'] == 2).astype(float)  # 0=Urbain(réf), 1=Rural
df['region_nord']     = df['v024'].isin([1, 4, 6]).astype(float)

#  Nominale v313 : contraceptif → 2 dummies (réf = Aucune méthode, code 0) 
df['contracep_trad']    = df['v313'].isin([1, 2]).astype(float)  # Folklorique ou Traditionnelle
df['contracep_moderne'] = (df['v313'] == 3).astype(float)        # Moderne

#  Nominale v501 : statut matrimonial → 5 dummies (réf = Jamais mariée, code 0) 
df['mariee']      = (df['v501'] == 1).astype(float)
df['union_libre'] = (df['v501'] == 2).astype(float)
df['veuve']       = (df['v501'] == 3).astype(float)
df['divorcee']    = (df['v501'] == 4).astype(float)
df['separee']     = (df['v501'] == 5).astype(float)

#  Nominale v130 : religion → 4 dummies (réf = Catholique, code 1) 
# Codes EDSC Cameroun 2018 : 1=Catholique 2=Protestant 3=Autres chrétiens
#                             4=Musulman 5=Animiste 6=Sans religion 96=Autre
df['rel_protestant']   = (df['v130'] == 2).astype(float)
df['rel_autres_chret'] = (df['v130'] == 3).astype(float)   # Autres chrétiens
df['rel_musulman']     = (df['v130'] == 4).astype(float)
df['rel_autres']       = df['v130'].isin([5, 6, 96]).astype(float)  # Animiste+Sans+Autre

#  Enfants décédés (effet de remplacement) 
# v206 = fils décédés, v207 = filles décédées
# Justification : une femme ayant perdu un enfant tend à désirer en avoir davantage
for col in ['v206', 'v207']:
    if col not in df.columns:
        df[col] = 0.0
df['nb_enfants_deces'] = (df['v206'].fillna(0) + df['v207'].fillna(0)).astype(float)

#  Âge au 1er mariage v511 
# ~36% de NA (femmes jamais mariées)
# → Décommenter uniquement pour une analyse restreinte aux femmes en union
# df['age_1er_mariage'] = df['v511'].astype(float)

#  Variable cible 
df['desir_enfant'] = df['v602'].apply(
    lambda x: 0 if x in [4,5,6,7,8] else (1 if x in [1,2,3] else np.nan))
df = df.dropna(subset=['desir_enfant'])
df['desir_enfant'] = df['desir_enfant'].astype(int)

N     = len(df)
N_OUI = int(df['desir_enfant'].sum())
N_NON = N - N_OUI

print(f'\n✓ ÉTAPE 1 terminée — encodage corrigé et étendu')
print(f'  N = {N:,} femmes (15-49 ans)')
print(f'  Désire         : {N_OUI:,} ({N_OUI/N*100:.1f}%)')
print(f'  Ne désire pas  : {N_NON:,} ({N_NON/N*100:.1f}%)')
print(f'\nVérification encodages :')
print(f'  contracep_trad    : {int(df["contracep_trad"].sum()):,} ({df["contracep_trad"].mean()*100:.1f}%)')
print(f'  contracep_moderne : {int(df["contracep_moderne"].sum()):,} ({df["contracep_moderne"].mean()*100:.1f}%)')
print(f'  mariee            : {int(df["mariee"].sum()):,} ({df["mariee"].mean()*100:.1f}%)')
print(f'  union_libre       : {int(df["union_libre"].sum()):,} ({df["union_libre"].mean()*100:.1f}%)')
print(f'  veuve             : {int(df["veuve"].sum()):,} ({df["veuve"].mean()*100:.1f}%)')
print(f'  divorcee          : {int(df["divorcee"].sum()):,} ({df["divorcee"].mean()*100:.1f}%)')
print(f'  separee           : {int(df["separee"].sum()):,} ({df["separee"].mean()*100:.1f}%)')
print(f'  residence_rural   : {int(df["residence_rural"].sum()):,} ({df["residence_rural"].mean()*100:.1f}%)')
print(f'\n --Religion (réf = Catholique) ')
print(f'  Catholique        : {int((df["v130"]==1).sum()):,} ({(df["v130"]==1).mean()*100:.1f}%)')
print(f'  rel_protestant    : {int(df["rel_protestant"].sum()):,} ({df["rel_protestant"].mean()*100:.1f}%)')
print(f'  rel_autres_chret  : {int(df["rel_autres_chret"].sum()):,} ({df["rel_autres_chret"].mean()*100:.1f}%)')
print(f'  rel_musulman      : {int(df["rel_musulman"].sum()):,} ({df["rel_musulman"].mean()*100:.1f}%)')
print(f'  rel_autres        : {int(df["rel_autres"].sum()):,} ({df["rel_autres"].mean()*100:.1f}%)')
print(f'\n --Effet de remplacement (nb_enfants_deces) :')
print(f'  nb_enfants_deces  : moy={df["nb_enfants_deces"].mean():.3f}  max={int(df["nb_enfants_deces"].max())}')

Chargement de CMIR71FL.SAV...
  Données brutes : 14,677 femmes · 5102 variables

✓ ÉTAPE 1 terminée — encodage corrigé et étendu
  N = 13,527 femmes (15-49 ans)
  Désire         : 12,782 (94.5%)
  Ne désire pas  : 745 (5.5%)

Vérification encodages :
  contracep_trad    : 437 (3.2%)
  contracep_moderne : 2,303 (17.0%)
  mariee            : 5,514 (40.8%)
  union_libre       : 1,949 (14.4%)
  veuve             : 364 (2.7%)
  divorcee          : 198 (1.5%)
  separee           : 646 (4.8%)
  residence_rural   : 6,184 (45.7%)

 --Religion (réf = Catholique) 
  Catholique        : 5,061 (37.4%)
  rel_protestant    : 3,877 (28.7%)
  rel_autres_chret  : 1,073 (7.9%)
  rel_musulman      : 3,059 (22.6%)
  rel_autres        : 235 (1.7%)

 --Effet de remplacement (nb_enfants_deces) :
  nb_enfants_deces  : moy=0.267  max=9


In [21]:

# DIAGNOSTIC PRÉ-MODÉLISATION
# Vérifie la multicolinéarité et les effectifs religion
# Décide automatiquement quelles variables inclure

print('═' * 62)
print('DIAGNOSTIC PRÉ-MODÉLISATION')
print('═' * 62)

# 1. Multicolinéarité : nb_enfants vs nb_enfants_deces 
print('\n POINT 1 — Corrélation : nb_enfants vs nb_enfants_deces')
corr_matrix = df[['nb_enfants', 'nb_enfants_deces']].corr()
r = corr_matrix.loc['nb_enfants', 'nb_enfants_deces']
print(f'   Pearson r = {r:.4f}')
print()

if r > 0.70:
    INCLURE_DECES = False
    FUSIONNER     = False
    DECISION_DECES = f'RETIRÉE  (r={r:.3f} > 0.70 → multicolinéarité élevée)'
    print(f'     r > 0.70 → multicolinéarité élevée')
    print(f'   → Décision : nb_enfants_deces RETIRÉE du modèle')
elif r > 0.50:
    INCLURE_DECES = False
    FUSIONNER     = True
    df['nb_enfants_total_mod'] = df['nb_enfants'] + df['nb_enfants_deces']
    DECISION_DECES = f'FUSIONNÉE en nb_enfants_total_mod (r={r:.3f}, entre 0.50 et 0.70)'
    print(f'     r entre 0.50 et 0.70 → multicolinéarité modérée')
    print(f'   → Décision : fusionner nb_enfants + nb_enfants_deces = nb_enfants_total_mod')
else:
    INCLURE_DECES = True
    FUSIONNER     = False
    DECISION_DECES = f'CONSERVÉE (r={r:.3f} < 0.50 → pas de problème)'
    print(f'   ✓  r < 0.50 → pas de problème de multicolinéarité')
    print(f'   → Décision : nb_enfants_deces CONSERVÉE (justification : effet de remplacement)')

#  2. Effectifs par catégorie de religion 
print('\n POINT 2 — Religion : effectifs par catégorie (réf = Catholique)')
religion_n = {
    'Catholique (réf)' : int((df['v130'] == 1).sum()),
    'Protestant'        : int(df['rel_protestant'].sum()),
    'Autres chrétiens'  : int(df['rel_autres_chret'].sum()),
    'Musulman'          : int(df['rel_musulman'].sum()),
    'Autres religions'  : int(df['rel_autres'].sum()),
}
RELIGION_OK = True
for cat, n_cat in religion_n.items():
    pct    = n_cat / N * 100
    statut = '✓' if n_cat >= 50 else '⚠ EFFECTIF FAIBLE — risque quasi-séparation'
    print(f'   {cat:<25} : {n_cat:>5} ({pct:4.1f}%)  {statut}')
    if n_cat < 50 and cat != 'Catholique (réf)':
        RELIGION_OK = False

if RELIGION_OK:
    print('\n   ✓ Toutes les catégories ≥ 50 obs. → les 4 dummies sont valides')
else:
    print('\n    Catégorie(s) < 50 obs. → déjà regroupées dans rel_autres → OK')

#  3. Bilan 
print('\n' + '─' * 62)
print('DÉCISION FINALE')
print('─' * 62)
print(f'   nb_enfants_deces : {DECISION_DECES}')
if FUSIONNER:
    print(f'   → nb_enfants_total_mod remplace nb_enfants dans le modèle')
print(f'   religion         : 4 dummies, réf = Catholique ✓')
print('─' * 62)

══════════════════════════════════════════════════════════════
DIAGNOSTIC PRÉ-MODÉLISATION
══════════════════════════════════════════════════════════════

 POINT 1 — Corrélation : nb_enfants vs nb_enfants_deces
   Pearson r = 0.3225

   ✓  r < 0.50 → pas de problème de multicolinéarité
   → Décision : nb_enfants_deces CONSERVÉE (justification : effet de remplacement)

 POINT 2 — Religion : effectifs par catégorie (réf = Catholique)
   Catholique (réf)          :  5061 (37.4%)  ✓
   Protestant                :  3877 (28.7%)  ✓
   Autres chrétiens          :  1073 ( 7.9%)  ✓
   Musulman                  :  3059 (22.6%)  ✓
   Autres religions          :   235 ( 1.7%)  ✓

   ✓ Toutes les catégories ≥ 50 obs. → les 4 dummies sont valides

──────────────────────────────────────────────────────────────
DÉCISION FINALE
──────────────────────────────────────────────────────────────
   nb_enfants_deces : CONSERVÉE (r=0.322 < 0.50 → pas de problème)
   religion         : 4 dummies, réf = Catholiq

In [22]:
# ÉTAPE 2 — Construction de VARS_MODELE (adapté au diagnostic)


# Labels de toutes les variables possibles
LABELS = {
    'const'                : 'Constante',
    'age'                  : 'Âge',
    'instruction'          : "Niveau d'instruction",
    'nb_enfants'           : 'Nb enfants vivants',
    'nb_enfants_deces'     : 'Nb enfants décédés',
    'nb_enfants_total_mod' : 'Nb enfants (vivants + décédés)',
    'contracep_trad'       : 'Contraceptif traditionnel',
    'contracep_moderne'    : 'Contraceptif moderne',
    'mariee'               : 'Mariée (vs jamais mariée)',
    'union_libre'          : 'Union libre (vs jamais mariée)',
    'veuve'                : 'Veuve (vs jamais mariée)',
    'divorcee'             : 'Divorcée (vs jamais mariée)',
    'separee'              : 'Séparée (vs jamais mariée)',
    'residence_rural'      : 'Résidence rurale (vs urbaine)',
    'quintile'             : 'Quintile de richesse',
    'emploi'               : 'Emploi (travail)',
    'region_nord'          : 'Région septentrionale',
    'rel_protestant'       : 'Protestante (vs Catholique)',
    'rel_autres_chret'     : 'Autres chrétiennes (vs Catholique)',
    'rel_musulman'         : 'Musulmane (vs Catholique)',
    'rel_autres'           : 'Autres religions (vs Catholique)',
}

#  Construction dynamique selon le diagnostic 
if FUSIONNER:
    # r entre 0.50 et 0.70 → on fusionne nb_enfants + nb_enfants_deces
    var_enfants = ['nb_enfants_total_mod']
elif INCLURE_DECES:
    # r < 0.50 → on garde les deux séparément
    var_enfants = ['nb_enfants', 'nb_enfants_deces']
else:
    # r > 0.70 → on retire nb_enfants_deces
    var_enfants = ['nb_enfants']

VARS_MODELE = (
    ['age', 'instruction'] +
    var_enfants +
    ['contracep_trad', 'contracep_moderne',
     'mariee', 'union_libre', 'veuve', 'divorcee', 'separee',
     'residence_rural', 'quintile', 'emploi', 'region_nord',
     'rel_protestant', 'rel_autres_chret', 'rel_musulman', 'rel_autres']
)

#  Matrices X 
df_model = df[VARS_MODELE + ['desir_enfant']].dropna().copy()
X  = df_model[VARS_MODELE].astype(float)
y  = df_model['desir_enfant'].astype(float)

X_sm = sm.add_constant(X)   # Pour statsmodels
X_sk = X.values              # Pour scikit-learn
y_sk = y.values

print('✓ ÉTAPE 2 terminée')
print(f'  Effectif modèle  : {len(df_model):,} observations')
print(f'  X shape          : {X.shape}  ({len(VARS_MODELE)} variables)')
print(f'\n  Variables dans VARS_MODELE :')
for i, v in enumerate(VARS_MODELE, 1):
    print(f'    {i:>2}. {v:<25}  → {LABELS[v]}')

✓ ÉTAPE 2 terminée
  Effectif modèle  : 13,527 observations
  X shape          : (13527, 19)  (19 variables)

  Variables dans VARS_MODELE :
     1. age                        → Âge
     2. instruction                → Niveau d'instruction
     3. nb_enfants                 → Nb enfants vivants
     4. nb_enfants_deces           → Nb enfants décédés
     5. contracep_trad             → Contraceptif traditionnel
     6. contracep_moderne          → Contraceptif moderne
     7. mariee                     → Mariée (vs jamais mariée)
     8. union_libre                → Union libre (vs jamais mariée)
     9. veuve                      → Veuve (vs jamais mariée)
    10. divorcee                   → Divorcée (vs jamais mariée)
    11. separee                    → Séparée (vs jamais mariée)
    12. residence_rural            → Résidence rurale (vs urbaine)
    13. quintile                   → Quintile de richesse
    14. emploi                     → Emploi (travail)
    15. region_nord       

In [23]:
# ÉTAPE 3 — Régression logistique (statsmodels)

print('Estimation du modèle logit (maximum de vraisemblance)...')
logit_model = Logit(y, X_sm).fit(disp=False)

# a) Paramètres
params = logit_model.params
conf   = logit_model.conf_int()
pvals  = logit_model.pvalues

# b) Odds Ratios + IC 95%
OR_vals = np.exp(params)
IC_inf  = np.exp(conf[0])
IC_sup  = np.exp(conf[1])

# c) Pseudo R² de Nagelkerke
n       = int(logit_model.nobs)
llnull  = logit_model.llnull
llf     = logit_model.llf
R2_CS   = 1 - np.exp((2/n) * (llnull - llf))          # Cox-Snell
R2_nag  = R2_CS / (1 - np.exp((2/n) * llnull))        # Nagelkerke
mcfad   = logit_model.prsquared

# d) Tableau : Variable | β | OR | IC_inf | IC_sup | p-value | Sig.
print('\n' + '='*85)
print('TABLE — ODDS RATIOS AJUSTÉS (encodage corrigé, v313 dummies + v501 dummies)')
print('='*85)
print(f'{"Variable":<33} {"β":>8} {"OR":>8} {"IC inf.":>9} {"IC sup.":>9} {"p-value":>10} {"Sig.":>5}')
print('-'*85)

or_rows = []
for var in params.index:
    p   = pvals[var]
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'NS'))
    lbl = LABELS.get(var, var)
    print(f'{lbl:<33} {params[var]:>8.4f} {OR_vals[var]:>8.4f} {IC_inf[var]:>9.4f} {IC_sup[var]:>9.4f} {p:>10.4f} {sig:>5}')
    or_rows.append({'Variable': lbl, 'β': round(params[var], 4),
                    'OR': round(OR_vals[var], 4), 'IC inf.': round(IC_inf[var], 4),
                    'IC sup.': round(IC_sup[var], 4), 'p-value': round(p, 6), 'Sig.': sig})

or_df = pd.DataFrame(or_rows)

# e) Export CSV
or_df.to_csv('../outputs/tables/04_odds_ratio_corriges.csv', index=False)

# f) Métriques de performance
y_pred_prob_rl = logit_model.predict(X_sm).values
y_pred_class   = (y_pred_prob_rl >= 0.5).astype(int)
cm_rl          = confusion_matrix(y.astype(int), y_pred_class)
tn_rl, fp_rl, fn_rl, tp_rl = cm_rl.ravel()
sensib_rl = tp_rl / (tp_rl + fn_rl) * 100
specif_rl = tn_rl / (tn_rl + fp_rl) * 100 if (tn_rl + fp_rl) > 0 else 0
pct_ok_rl = (tp_rl + tn_rl) / n * 100
auc_rl    = roc_auc_score(y.astype(int), y_pred_prob_rl)

print('\n' + '='*55)
print('INDICATEURS DU MODÈLE LOGISTIQUE')
print('='*55)
print(f'  N                    : {n:,}')
print(f'  AIC                  : {logit_model.aic:.2f}')
print(f'  BIC                  : {logit_model.bic:.2f}')
print(f'  LLR p-value          : {logit_model.llr_pvalue:.2e}')
print(f'  Pseudo R² McFadden   : {mcfad:.4f} ({mcfad*100:.1f}%)')
print(f'  Pseudo R² Nagelkerke : {R2_nag:.4f} ({R2_nag*100:.1f}%)')
print(f'  AUC-ROC              : {auc_rl:.4f}')
print(f'  % bien classés       : {pct_ok_rl:.1f}%')
print(f'  Sensibilité          : {sensib_rl:.1f}%')
print(f'  Spécificité          : {specif_rl:.1f}%')
print(f'\n✓ CSV sauvegardé → outputs/tables/04_odds_ratio_corriges.csv')

Estimation du modèle logit (maximum de vraisemblance)...

TABLE — ODDS RATIOS AJUSTÉS (encodage corrigé, v313 dummies + v501 dummies)
Variable                                 β       OR   IC inf.   IC sup.    p-value  Sig.
-------------------------------------------------------------------------------------
Constante                           6.7785 878.7264  522.5498 1477.6774     0.0000   ***
Âge                                -0.1517   0.8592    0.8496    0.8690     0.0000   ***
Niveau d'instruction                0.1436   1.1544    1.0167    1.3108     0.0267     *
Nb enfants vivants                  0.1828   1.2006    1.1528    1.2503     0.0000   ***
Nb enfants décédés                  0.0248   1.0251    0.9422    1.1153     0.5640    NS
Contraceptif traditionnel           1.9639   7.1272    2.6199   19.3891     0.0001   ***
Contraceptif moderne                0.3897   1.4765    1.1342    1.9220     0.0038    **
Mariée (vs jamais mariée)           0.3664   1.4426    1.1120    1.8

In [24]:
# DIAGNOSTIC 1 — VIF (Variance Inflation Factor)
# Détecte la multicolinéarité entre les variables explicatives
# Règle : VIF < 5 = OK · 5-10 = modéré · > 10 = problème

from statsmodels.stats.outliers_influence import variance_inflation_factor

print('VIF — Facteur d\'inflation de la variance (modèle corrigé)\n')

vif_rows = []
for i, var in enumerate(VARS_MODELE):
    vif_val = variance_inflation_factor(X.values, i)
    lbl     = LABELS.get(var, var)
    if vif_val < 5:
        statut = 'OK'
        flag   = ''
    elif vif_val < 10:
        statut = 'Modéré'
        flag   = '!'
    else:
        statut = 'PROBLÈME'
        flag   = 'X'
    vif_rows.append({'Variable': lbl, 'Code': var,
                     'VIF': round(vif_val, 3), 'Statut': statut})

vif_df = pd.DataFrame(vif_rows).sort_values('VIF', ascending=False).reset_index(drop=True)

print(f'{"Variable":<38} {"VIF":>8}   Statut')
print('-' * 58)
for _, row in vif_df.iterrows():
    flag = 'X' if row['Statut'] == 'PROBLÈME' else ('!' if row['Statut'] == 'Modéré' else 'OK')
    print(f'{row["Variable"]:<38} {row["VIF"]:>8.3f}   [{flag}] {row["Statut"]}')

# Export
vif_df.to_csv('../outputs/tables/05_vif_corriges.csv', index=False)

# Résumé
n_pb  = (vif_df['Statut'] == 'PROBLÈME').sum()
n_mod = (vif_df['Statut'] == 'Modéré').sum()
n_ok  = (vif_df['Statut'] == 'OK').sum()
print(f'\n  OK (<5)       : {n_ok} variables')
print(f'  Modéré (5-10) : {n_mod} variables')
print(f'  Problème (>10): {n_pb} variables')

if n_pb > 0:
    vars_pb = vif_df[vif_df['Statut'] == 'PROBLÈME']['Variable'].tolist()
    print(f'\n  [X] Variables à surveiller : {vars_pb}')
    print('      → Envisager de retirer l\'une d\'elles ou de fusionner')
else:
    print('\n  ✓ Pas de multicolinéarité sévère dans le modèle corrigé')

print('\n✓ Sauvegardé → outputs/tables/05_vif_corriges.csv')

VIF — Facteur d'inflation de la variance (modèle corrigé)

Variable                                    VIF   Statut
----------------------------------------------------------
Âge                                      19.430   [X] PROBLÈME
Quintile de richesse                     11.975   [X] PROBLÈME
Niveau d'instruction                      7.100   [!] Modéré
Nb enfants vivants                        4.550   [OK] OK
Mariée (vs jamais mariée)                 3.806   [OK] OK
Emploi (travail)                          3.062   [OK] OK
Résidence rurale (vs urbaine)             2.241   [OK] OK
Musulmane (vs Catholique)                 1.806   [OK] OK
Protestante (vs Catholique)               1.703   [OK] OK
Union libre (vs jamais mariée)            1.673   [OK] OK
Nb enfants décédés                        1.365   [OK] OK
Région septentrionale                     1.317   [OK] OK
Contraceptif moderne                      1.295   [OK] OK
Veuve (vs jamais mariée)                  1.288   [OK] OK


In [25]:
# DIAGNOSTIC 2 — Test de Hosmer-Lemeshow
# Évalue l'adéquation globale du modèle logistique
# H0 : le modèle s'ajuste bien aux données
# → p > 0.05 : bon ajustement  /  p ≤ 0.05 : ajustement à vérifier

print('Test de Hosmer-Lemeshow (adéquation du modèle logistique)\n')

# Découper en 10 déciles de probabilité prédite
deciles = pd.qcut(y_pred_prob_rl, q=10, duplicates='drop')
hl_df   = pd.DataFrame({
    'obs' : y.values.astype(int),
    'pred': y_pred_prob_rl,
    'dec' : deciles
})

hl_table = hl_df.groupby('dec', observed=True).agg(
    obs1      = ('obs',  'sum'),
    n         = ('obs',  'count'),
    pred_mean = ('pred', 'mean')
).reset_index()

hl_table['expected1'] = hl_table['n'] * hl_table['pred_mean']
hl_table['expected0'] = hl_table['n'] * (1 - hl_table['pred_mean'])
hl_table['obs0']      = hl_table['n'] - hl_table['obs1']

# Statistique chi² de Hosmer-Lemeshow
chi2_hl = (
    ((hl_table['obs1'] - hl_table['expected1']) ** 2 / hl_table['expected1'].clip(lower=1e-6)) +
    ((hl_table['obs0'] - hl_table['expected0']) ** 2 / hl_table['expected0'].clip(lower=1e-6))
).sum()

ddl_hl = len(hl_table) - 2   # ddl = G - 2 (G = nombre de groupes)
p_hl   = 1 - chi2_dist.cdf(chi2_hl, df=ddl_hl)

# Affichage de la table détaillée
print(f'  {"Décile":<20} {"n":>6} {"Obs(1)":>8} {"Exp(1)":>8} {"Obs(0)":>8} {"Exp(0)":>8}')
print('  ' + '-' * 62)
for _, row in hl_table.iterrows():
    print(f'  {str(row["dec"]):<20} {int(row["n"]):>6} {int(row["obs1"]):>8} '
          f'{row["expected1"]:>8.1f} {int(row["obs0"]):>8} {row["expected0"]:>8.1f}')

print(f'\n  Hosmer-Lemeshow : χ²({ddl_hl}) = {chi2_hl:.4f}   p = {p_hl:.4f}')
print()

if p_hl > 0.05:
    print('  ✓ p > 0.05 → Bon ajustement du modèle (H0 non rejetée)')
    print('    Le modèle prédit correctement les probabilités observées.')
else:
    print('  [!] p ≤ 0.05 → Ajustement imparfait (H0 rejetée)')
    print('    IMPORTANT : avec n = {:,}, ce test est très sensible à la taille'.format(n))
    print('    de l\'échantillon. Un p faible peut refléter la puissance du test')
    print('    plutôt qu\'un vrai mauvais ajustement.')
    print('    → Compléter l\'interprétation avec la courbe de calibration (Fig A2)')
    print('      et le Brier score pour une évaluation plus nuancée.')

# Export indicateurs dans le CSV existant
indic_hl = pd.DataFrame([
    {'Indicateur': 'HL Chi²',   'Valeur': round(chi2_hl, 4)},
    {'Indicateur': 'HL ddl',    'Valeur': ddl_hl},
    {'Indicateur': 'HL p-value','Valeur': round(p_hl, 6)},
    {'Indicateur': 'HL groupes','Valeur': len(hl_table)},
])
indic_hl.to_csv('../outputs/tables/04b_hosmer_lemeshow.csv', index=False)
print('\n✓ Sauvegardé → outputs/tables/04b_hosmer_lemeshow.csv')

Test de Hosmer-Lemeshow (adéquation du modèle logistique)

  Décile                    n   Obs(1)   Exp(1)   Obs(0)   Exp(0)
  --------------------------------------------------------------
  (0.305, 0.849]         1353      969    996.9      384    356.1
  (0.849, 0.926]         1353     1233   1208.4      120    144.6
  (0.926, 0.955]         1352     1284   1274.1       68     77.9
  (0.955, 0.97]          1353     1316   1303.4       37     49.6
  (0.97, 0.978]          1353     1329   1318.2       24     34.8
  (0.978, 0.983]         1353     1330   1327.2       23     25.8
  (0.983, 0.987]         1352     1328   1332.2       24     19.8
  (0.987, 0.99]          1352     1331   1336.2       21     15.8
  (0.99, 0.992]          1353     1329   1340.5       24     12.5
  (0.992, 0.999]         1353     1333   1344.9       20      8.1

  Hosmer-Lemeshow : χ²(8) = 47.0896   p = 0.0000

  [!] p ≤ 0.05 → Ajustement imparfait (H0 rejetée)
    IMPORTANT : avec n = 13,527, ce test est trè

In [26]:

# ÉTAPE 4 — Préparation : Split + SMOTE + Scaler

print('ÉTAPE 4 — Split 80/20 stratifié...')

X_train, X_test, y_train, y_test = train_test_split(
    X_sk, y_sk, test_size=0.2, random_state=42, stratify=y_sk)
print(f'  Train : {len(X_train):,} obs. → Classe 1 : {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'  Test  : {len(X_test):,} obs.  → Classe 1 : {y_test.sum():,} ({y_test.mean()*100:.1f}%)')

# SMOTE sur le train uniquement (JAMAIS sur le test)
print('\nSMOTE sur le train uniquement...')
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f'  Après SMOTE : {len(X_train_sm):,} obs. (50%/50%)')
print(f'  Observations synthétiques créées : {len(X_train_sm)-len(X_train):,}')

# StandardScaler pour SVM et KNN uniquement
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)  # fit sur train
X_test_sc  = scaler.transform(X_test)          # transform sur test
print('\n✓ Normalisation (StandardScaler) → utilisée pour SVM et KNN uniquement')

# Définition des 6 modèles
modeles = {
    'Rég. Logistique' : LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'   : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Arbre Décision'  : DecisionTreeClassifier(max_depth=5, random_state=42),
    'SVM (RBF)'       : SVC(kernel='rbf', probability=True, random_state=42),
    'KNN (k=5)'       : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'XGBoost'         : XGBClassifier(n_estimators=100, random_state=42,
                                       eval_metric='logloss', verbosity=0),
}
print('\n✓ 6 modèles définis :', list(modeles.keys()))

ÉTAPE 4 — Split 80/20 stratifié...
  Train : 10,821 obs. → Classe 1 : 10,225.0 (94.5%)
  Test  : 2,706 obs.  → Classe 1 : 2,557.0 (94.5%)

SMOTE sur le train uniquement...
  Après SMOTE : 20,450 obs. (50%/50%)
  Observations synthétiques créées : 9,629

✓ Normalisation (StandardScaler) → utilisée pour SVM et KNN uniquement

✓ 6 modèles définis : ['Rég. Logistique', 'Random Forest', 'Arbre Décision', 'SVM (RBF)', 'KNN (k=5)', 'XGBoost']


In [27]:
# Entraînement + évaluation des 6 modèles
resultats    = []
roc_data     = {}
auc_folds    = {}  # 5 AUC individuels par fold (pour boxplot)
brier_scores = {}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'{"Modèle":<18} {"Acc":>7} {"AUC_test":>9} {"F1":>7} {"Sensib":>8} {"Specif":>8} {"AUC_CV5":>9} {"Std":>7} {"Gap":>7}')
print('-' * 88)

for nom, clf in modeles.items():
    use_scaled = nom in ['SVM (RBF)', 'KNN (k=5)']
    Xtr = X_train_sc if use_scaled else X_train_sm
    Xte = X_test_sc  if use_scaled else X_test

    clf.fit(Xtr, y_train_sm)

    y_pred  = clf.predict(Xte)
    y_proba = clf.predict_proba(Xte)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)
    f1   = f1_score(y_test, y_pred)
    brier = brier_score_loss(y_test, y_proba)

    cm_i = confusion_matrix(y_test, y_pred)
    tn_i, fp_i, fn_i, tp_i = cm_i.ravel()
    sens_i = tp_i / (tp_i + fn_i) if (tp_i + fn_i) > 0 else 0.0
    spec_i = tn_i / (tn_i + fp_i) if (tn_i + fp_i) > 0 else 0.0

    auc_train = roc_auc_score(y_train_sm, clf.predict_proba(Xtr)[:, 1])
    gap = auc_train - auc

    cv_scores = cross_val_score(clf, Xtr, y_train_sm, cv=skf, scoring='roc_auc', n_jobs=-1)
    auc_cv5 = cv_scores.mean()
    auc_std = cv_scores.std()

    print(f'{nom:<18} {acc:>7.4f} {auc:>9.4f} {f1:>7.4f} {sens_i:>8.4f} {spec_i:>8.4f} {auc_cv5:>9.4f} {auc_std:>7.4f} {gap:>7.4f}')

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[nom] = {'fpr': fpr, 'tpr': tpr, 'auc': auc}
    auc_folds[nom]    = cv_scores
    brier_scores[nom] = brier

    if gap > 0.20:
        diag = 'Overfitting sévère'
    elif gap > 0.10:
        diag = 'Léger overfitting'
    elif auc < 0.65:
        diag = 'Modèle insuffisant'
    else:
        diag = 'Bon modèle'

    resultats.append({
        'Modèle': nom, 'Accuracy': round(acc,4), 'AUC_test': round(auc,4),
        'F1': round(f1,4), 'Sensibilité': round(sens_i,4), 'Spécificité': round(spec_i,4),
        'AUC_train': round(auc_train,4), 'AUC_CV5': round(auc_cv5,4),
        'Std_CV5': round(auc_std,4), 'Gap': round(gap,4),
        'Brier': round(brier,4), 'Diagnostic': diag,
        'cm': cm_i, 'y_proba_test': y_proba
    })

print('\n✓ Entraînement et évaluation terminés')

Modèle                 Acc  AUC_test      F1   Sensib   Specif   AUC_CV5     Std     Gap
----------------------------------------------------------------------------------------
Rég. Logistique     0.7857    0.8198  0.8740   0.7869   0.7651    0.8182  0.0043 -0.0007
Random Forest       0.9342    0.7757  0.9658   0.9828   0.1007    0.9926  0.0007  0.2241
Arbre Décision      0.8514    0.7443  0.9171   0.8694   0.5436    0.8939  0.0023  0.1570
SVM (RBF)           0.9161    0.7949  0.9551   0.9433   0.4497    0.9693  0.0018  0.1811
KNN (k=5)           0.8684    0.6374  0.9284   0.9022   0.2886    0.9756  0.0006  0.3580
XGBoost             0.9313    0.7961  0.9641   0.9769   0.1477    0.9884  0.0012  0.2013

✓ Entraînement et évaluation terminés


In [28]:
# Export CSV comparaison
res_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('cm', 'y_proba_test')}
                        for r in resultats])
res_df = res_df.sort_values('AUC_CV5', ascending=False).reset_index(drop=True)
res_df.to_csv('../outputs/tables/06_comparaison_modeles_corriges.csv', index=False)

print('TABLEAU COMPARATIF — 6 MODÈLES ML (après correction encodage)')
print('='*72)
cols_aff = ['Modèle','AUC_test','AUC_CV5','Std_CV5','Gap','Brier','Diagnostic']
print(res_df[cols_aff].to_string(index=False))
print('\n✓ Sauvegardé → outputs/tables/06_comparaison_modeles_corriges.csv')

# Sélection du meilleur modèle
best_row = res_df[res_df['Gap'] <= 0.20].sort_values('AUC_CV5', ascending=False).iloc[0]
best_nom = best_row['Modèle']
best_clf = modeles[best_nom]
use_sc   = best_nom in ['SVM (RBF)', 'KNN (k=5)']
Xte_best = X_test_sc if use_sc else X_test
y_proba_best = best_clf.predict_proba(Xte_best)[:, 1]
y_pred_best  = best_clf.predict(Xte_best)

print(f'\n Modèle retenu : {best_nom}')
print(f'   AUC CV-5 = {best_row["AUC_CV5"]:.4f} | AUC test = {best_row["AUC_test"]:.4f} | Gap = {best_row["Gap"]:.4f}')

TABLEAU COMPARATIF — 6 MODÈLES ML (après correction encodage)
         Modèle  AUC_test  AUC_CV5  Std_CV5     Gap  Brier         Diagnostic
  Random Forest    0.7757   0.9926   0.0007  0.2241 0.0526 Overfitting sévère
        XGBoost    0.7961   0.9884   0.0012  0.2013 0.0528 Overfitting sévère
      KNN (k=5)    0.6374   0.9756   0.0006  0.3580 0.1028 Overfitting sévère
      SVM (RBF)    0.7949   0.9693   0.0018  0.1811 0.0742  Léger overfitting
 Arbre Décision    0.7443   0.8939   0.0023  0.1570 0.1164  Léger overfitting
Rég. Logistique    0.8198   0.8182   0.0043 -0.0007 0.1622         Bon modèle

✓ Sauvegardé → outputs/tables/06_comparaison_modeles_corriges.csv

 Modèle retenu : SVM (RBF)
   AUC CV-5 = 0.9693 | AUC test = 0.7949 | Gap = 0.1811


In [29]:
# ════════════════════════════════════════════════════════
# ÉTAPE 5 — Validation Partie A : Régression Logistique
# Planche 2×2 : ROC | Calibration | Résidus Pearson | Sensib/Specif vs Seuil
# ════════════════════════════════════════════════════════
print('ÉTAPE 5 — Génération de la figure de validation logistique...')

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(
    'Validation du modèle de Régression Logistique\nEDSC Cameroun 2018 (encodage corrigé)',
    fontsize=14, fontweight='bold', y=1.01
)

# ── A1 : Courbe ROC ─────────────────────────────────────
ax = axes[0, 0]
fpr_rl, tpr_rl, _ = roc_curve(y.astype(int), y_pred_prob_rl)
ax.plot(fpr_rl, tpr_rl, color=NAVY, linewidth=2.5,
        label=f'Régression Logistique (AUC = {auc_rl:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Modèle aléatoire (AUC = 0.500)')
ax.fill_between(fpr_rl, tpr_rl, alpha=0.08, color=NAVY)
ax.set_xlabel('Taux de faux positifs (1 − Spécificité)', fontsize=10)
ax.set_ylabel('Taux de vrais positifs (Sensibilité)', fontsize=10)
ax.set_title('Fig A1 — Courbe ROC (Régression Logistique)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

# ── A2 : Calibration plot ────────────────────────────────
ax = axes[0, 1]
brier_rl = brier_score_loss(y.astype(int), y_pred_prob_rl)
frac_pos, mean_pred = calibration_curve(y.astype(int), y_pred_prob_rl, n_bins=10, strategy='uniform')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Calibration parfaite', alpha=0.7)
ax.plot(mean_pred, frac_pos, 'o-', color=TEAL, linewidth=2.2, markersize=7,
        label=f'Modèle (Brier = {brier_rl:.4f})')
ax.fill_between(mean_pred, frac_pos, mean_pred, alpha=0.1, color=TEAL)
ax.set_xlabel('Probabilité prédite moyenne', fontsize=10)
ax.set_ylabel('Fraction de cas positifs observés', fontsize=10)
ax.set_title(f'Fig A2 — Calibration (Brier = {brier_rl:.4f})', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# ── A3 : Résidus de Pearson ──────────────────────────────
ax = axes[1, 0]
p_pred = np.clip(y_pred_prob_rl, 1e-6, 1 - 1e-6)
residus_pearson = (y.values - p_pred) / np.sqrt(p_pred * (1 - p_pred))
outliers = np.abs(residus_pearson) > 2
couleurs = np.where(outliers, CORAL, GRAY)
ax.scatter(p_pred, residus_pearson, c=couleurs, alpha=0.4, s=8)
ax.axhline(0, color='black', linewidth=1.2)
ax.axhline(2, color=CORAL, linewidth=1, linestyle='--', alpha=0.8, label='Seuil ±2')
ax.axhline(-2, color=CORAL, linewidth=1, linestyle='--', alpha=0.8)
n_outliers = outliers.sum()
ax.set_xlabel('Probabilité prédite', fontsize=10)
ax.set_ylabel('Résidu de Pearson', fontsize=10)
ax.set_title(f'Fig A3 — Résidus de Pearson ({n_outliers:,} outliers > |2|)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.text(0.98, 0.98, f'{n_outliers:,} outliers\n({n_outliers/n*100:.1f}%)',
        ha='right', va='top', transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor=CORAL, alpha=0.2))

# ── A4 : Sensibilité / Spécificité vs Seuil ─────────────
ax = axes[1, 1]
seuils   = np.arange(0, 1, 0.01)
sensib_s = []
specif_s = []
for s in seuils:
    y_s = (y_pred_prob_rl >= s).astype(int)
    cm_s = confusion_matrix(y.astype(int), y_s, labels=[0, 1])
    tn_s, fp_s, fn_s, tp_s = cm_s.ravel()
    sensib_s.append(tp_s / (tp_s + fn_s) if (tp_s + fn_s) > 0 else 0)
    specif_s.append(tn_s / (tn_s + fp_s) if (tn_s + fp_s) > 0 else 0)

sensib_s = np.array(sensib_s)
specif_s = np.array(specif_s)

# Seuil optimal = intersection sensib / specif
diff_abs = np.abs(sensib_s - specif_s)
seuil_opt = seuils[np.argmin(diff_abs)]
val_opt   = (sensib_s[np.argmin(diff_abs)] + specif_s[np.argmin(diff_abs)]) / 2

ax.plot(seuils, sensib_s, color=BLUE,  linewidth=2.2, label='Sensibilité')
ax.plot(seuils, specif_s, color=CORAL, linewidth=2.2, label='Spécificité')
ax.axvline(seuil_opt, color=GREEN, linewidth=1.8, linestyle='-.',
           label=f'Seuil optimal ({seuil_opt:.2f})')
ax.axvline(0.5, color=GRAY, linewidth=1.2, linestyle='--', alpha=0.7,
           label='Seuil par défaut (0.50)')
ax.plot(seuil_opt, val_opt, 'o', color=GREEN, markersize=9, zorder=5)
ax.set_xlabel('Seuil de décision', fontsize=10)
ax.set_ylabel('Score', fontsize=10)
ax.set_title(f'Fig A4 — Sensibilité & Spécificité vs Seuil\nIntersection à {seuil_opt:.2f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../outputs/figures/figA_validation_logistique.png', bbox_inches='tight', dpi=130)
plt.show()
print('✓ ÉTAPE 5 terminée — figure sauvegardée : figA_validation_logistique.png')
print(f'  AUC       : {auc_rl:.4f}')
print(f'  Brier     : {brier_rl:.4f}')
print(f'  Outliers  : {n_outliers:,} ({n_outliers/n*100:.1f}%)')
print(f'  Seuil opt : {seuil_opt:.2f}')

ÉTAPE 5 — Génération de la figure de validation logistique...
✓ ÉTAPE 5 terminée — figure sauvegardée : figA_validation_logistique.png
  AUC       : 0.8090
  Brier     : 0.0445
  Outliers  : 438 (3.2%)
  Seuil opt : 0.95


In [30]:
# ════════════════════════════════════════════════════════
# LEARNING CURVES — ÉTAPE 1
# Calcul des learning curves pour les 6 modèles
# Données : X_train_sm / y_train_sm (après SMOTE, jamais X_test)
# ════════════════════════════════════════════════════════
print('ÉTAPE 1 — Calcul des learning curves (6 modèles × 10 tailles × CV-5)...')
print('Cela peut prendre 3 à 5 minutes — SVM est le plus lent.\n')

train_sizes_rel = np.linspace(0.1, 1.0, 10)  # 10% → 100% de X_train_sm
lc_resultats    = {}  # stockage : nom → dict de métriques

for nom, clf in modeles.items():
    # SVM et KNN → données normalisées (StandardScaler déjà fitté sur X_train_sm)
    use_sc = nom in ['SVM (RBF)', 'KNN (k=5)']
    Xtr_lc = X_train_sc if use_sc else X_train_sm

    sz_abs, tr_sc, val_sc = learning_curve(
        estimator   = clf,
        X           = Xtr_lc,
        y           = y_train_sm,
        train_sizes = train_sizes_rel,
        cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring     = 'roc_auc',
        n_jobs      = -1
    )

    tr_mean = tr_sc.mean(axis=1)
    tr_std  = tr_sc.std(axis=1)
    va_mean = val_sc.mean(axis=1)
    va_std  = val_sc.std(axis=1)

    gap_final = tr_mean[-1] - va_mean[-1]
    auc_val   = va_mean[-1]
    convergence = gap_final <= 0.10

    # Diagnostic automatique
    if gap_final > 0.20:
        diag_lc = 'Overfitting sévère'
        diag_icon = 'X'
    elif gap_final > 0.10:
        diag_lc = 'Léger overfitting'
        diag_icon = '!'
    elif auc_val < 0.60:
        diag_lc = 'Underfitting'
        diag_icon = 'X'
    elif convergence and auc_val >= 0.75:
        diag_lc = 'Bon modèle'
        diag_icon = 'OK'
    else:
        diag_lc = 'A surveiller'
        diag_icon = '!'

    lc_resultats[nom] = {
        'sz_abs'      : sz_abs,
        'tr_mean'     : tr_mean,
        'tr_std'      : tr_std,
        'va_mean'     : va_mean,
        'va_std'      : va_std,
        'gap_final'   : gap_final,
        'auc_val'     : auc_val,
        'auc_train'   : tr_mean[-1],
        'convergence' : convergence,
        'diagnostic'  : diag_lc,
        'icon'        : diag_icon,
    }
    print(f'  [{diag_icon:>2}] {nom:<18}  AUC_val={auc_val:.4f}  '
          f'AUC_train={tr_mean[-1]:.4f}  Gap={gap_final:.4f}  → {diag_lc}')

print('\n✓ Calcul terminé')

ÉTAPE 1 — Calcul des learning curves (6 modèles × 10 tailles × CV-5)...
Cela peut prendre 3 à 5 minutes — SVM est le plus lent.

  [OK] Rég. Logistique     AUC_val=0.8182  AUC_train=0.8193  Gap=0.0010  → Bon modèle
  [OK] Random Forest       AUC_val=0.9926  AUC_train=0.9999  Gap=0.0073  → Bon modèle
  [OK] Arbre Décision      AUC_val=0.8939  AUC_train=0.8991  Gap=0.0052  → Bon modèle
  [OK] SVM (RBF)           AUC_val=0.9693  AUC_train=0.9741  Gap=0.0048  → Bon modèle
  [OK] KNN (k=5)           AUC_val=0.9756  AUC_train=0.9946  Gap=0.0190  → Bon modèle
  [OK] XGBoost             AUC_val=0.9884  AUC_train=0.9979  Gap=0.0095  → Bon modèle

✓ Calcul terminé


In [31]:

# LEARNING CURVES — ÉTAPE 2
# Figure 2×3 : un subplot par modèle

print('ÉTAPE 2 — Figure 2×3 (6 subplots)...')

# Couleurs fixes par slot (indépendantes du modèle)
SLOT_COLORS = [NAVY, TEAL, CORAL, AMBER, GREEN, BLUE]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Learning Curves — 6 modèles ML (AUC-ROC vs taille entraînement)\n'
             'Données SMOTE · CV-5 stratifié · EDSC Cameroun 2018',
             fontsize=14, fontweight='bold')

for ax, (nom, lc), col in zip(axes.ravel(), lc_resultats.items(), SLOT_COLORS):
    sz      = lc['sz_abs']
    tr_m    = lc['tr_mean']
    tr_s    = lc['tr_std']
    va_m    = lc['va_mean']
    va_s    = lc['va_std']
    gap     = lc['gap_final']
    diag    = lc['diagnostic']

    # Courbe train
    ax.plot(sz, tr_m, 'o-', color=col, linewidth=2.2, label='AUC entraînement')
    ax.fill_between(sz, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color=col)

    # Courbe validation
    ax.plot(sz, va_m, 's--', color=AMBER, linewidth=2.2, label='AUC validation CV-5')
    ax.fill_between(sz, va_m - va_s, va_m + va_s, alpha=0.15, color=AMBER)

    # Lignes de référence
    ax.axhline(0.50, color=GRAY,  linewidth=1, linestyle=':', alpha=0.6, label='Aléatoire (0.50)')
    ax.axhline(0.75, color=GREEN, linewidth=1, linestyle='--', alpha=0.7, label='Seuil acceptable (0.75)')

    # Annotation gap
    ax.text(0.98, 0.05, f'Gap = {gap:.3f}',
            ha='right', va='bottom', transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))

    # Titre avec diagnostic
    icone = {'Bon modèle': '✓', 'Léger overfitting': '!',
             'Overfitting sévère': 'X', 'Underfitting': 'X', 'A surveiller': '!'}.get(diag, '?')
    retenu = '  ← RETENU' if nom == best_nom else ''
    ax.set_title(f'{nom}{retenu}\n[{icone}] {diag}',
                 fontsize=10, fontweight='bold',
                 color=GREEN if nom == best_nom else 'black')

    ax.set_xlabel("Taille d'entraînement (observations)", fontsize=9)
    ax.set_ylabel('AUC-ROC', fontsize=9)
    ax.set_ylim(0.40, 1.05)
    ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('../outputs/figures/fig_learning_curves_6modeles.png', bbox_inches='tight', dpi=130)
plt.show()
print('✓ Étape 2 terminée → fig_learning_curves_6modeles.png')

ÉTAPE 2 — Figure 2×3 (6 subplots)...
✓ Étape 2 terminée → fig_learning_curves_6modeles.png


In [32]:

# LEARNING CURVES — ÉTAPE 3
# Figure focus : SVM retenu vs Régression Logistique
# Zones colorées selon la convergence

print('ÉTAPE 3 — Figure focus : SVM vs Régression Logistique...')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Learning Curves — Analyse détaillée : Modèle retenu vs Référence\n'
             'Zone verte = convergence (gap < 0.10) · Zone rouge = overfitting (gap > 0.20)',
             fontsize=13, fontweight='bold')

FOCUS_PAIRS = [
    (best_nom,         NAVY,  'Modèle retenu'),
    ('Rég. Logistique', CORAL, 'Référence interprétable'),
]

for ax, (nom, col, role) in zip(axes, FOCUS_PAIRS):
    lc      = lc_resultats[nom]
    sz      = lc['sz_abs']
    tr_m    = lc['tr_mean']
    tr_s    = lc['tr_std']
    va_m    = lc['va_mean']
    va_s    = lc['va_std']
    gap     = lc['gap_final']
    auc_t   = lc['auc_train']
    auc_v   = lc['auc_val']

    # Zone colorée selon le gap à chaque point
    for i in range(len(sz) - 1):
        g_i = tr_m[i] - va_m[i]
        if g_i <= 0.10:
            zone_col = '#C8E6C9'  # vert pâle → convergence
        elif g_i <= 0.20:
            zone_col = '#FFF9C4'  # jaune pâle → léger overfitting
        else:
            zone_col = '#FFCDD2'  # rouge pâle → overfitting sévère
        ax.fill_betweenx([0.3, 1.05], sz[i], sz[i+1], color=zone_col, alpha=0.6)

    # Courbes
    ax.plot(sz, tr_m, 'o-', color=col,   linewidth=2.5, markersize=7, label='AUC entraînement')
    ax.fill_between(sz, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color=col)
    ax.plot(sz, va_m, 's--', color=AMBER, linewidth=2.5, markersize=7, label='AUC validation CV-5')
    ax.fill_between(sz, va_m - va_s, va_m + va_s, alpha=0.15, color=AMBER)

    # Références horizontales
    ax.axhline(0.50, color=GRAY,  linewidth=1, linestyle=':', alpha=0.5)
    ax.axhline(0.75, color=GREEN, linewidth=1.5, linestyle='--', alpha=0.8, label='Seuil 0.75')

    # Annotation finale (dernier point)
    ax.annotate(f'Train : {auc_t:.3f}',
                xy=(sz[-1], tr_m[-1]), xytext=(sz[-1]*0.82, tr_m[-1]+0.025),
                fontsize=9, color=col, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=col, lw=1.2))
    ax.annotate(f'Val   : {auc_v:.3f}',
                xy=(sz[-1], va_m[-1]), xytext=(sz[-1]*0.82, va_m[-1]-0.04),
                fontsize=9, color=AMBER, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=AMBER, lw=1.2))

    # Boîte bilan
    ax.text(0.03, 0.06,
            f'AUC train : {auc_t:.4f}\nAUC val   : {auc_v:.4f}\nGap       : {gap:.4f}\n{lc["diagnostic"]}',
            ha='left', va='bottom', transform=ax.transAxes, fontsize=9.5,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=col, linewidth=1.5))

    ax.set_title(f'{nom}  ({role})', fontsize=12, fontweight='bold', color=col)
    ax.set_xlabel("Taille d'entraînement (observations)", fontsize=10)
    ax.set_ylabel('AUC-ROC', fontsize=10)
    ax.set_ylim(0.35, 1.08)
    ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('../outputs/figures/fig_learning_curves_focus.png', bbox_inches='tight', dpi=130)
plt.show()
print('✓ Étape 3 terminée → fig_learning_curves_focus.png')

ÉTAPE 3 — Figure focus : SVM vs Régression Logistique...
✓ Étape 3 terminée → fig_learning_curves_focus.png


In [33]:
# ════════════════════════════════════════════════════════
# LEARNING CURVES — ÉTAPE 4 : Tableau récapitulatif
# + ÉTAPE 5 : Interprétation automatique dans le terminal
# ════════════════════════════════════════════════════════

# ── ÉTAPE 4 — Tableau ────────────────────────────────────
print('ÉTAPE 4 — Tableau récapitulatif des learning curves\n')

recap_rows = []
for nom, lc in lc_resultats.items():
    # Valeurs au 1er point (10% des données) et au dernier (100%)
    auc_tr_10  = lc['tr_mean'][0]
    auc_va_10  = lc['va_mean'][0]
    auc_tr_100 = lc['auc_train']
    auc_va_100 = lc['auc_val']
    gap        = lc['gap_final']
    conv       = 'Oui' if lc['convergence'] else 'Non'

    recap_rows.append({
        'Modèle'          : nom,
        'AUC_train_10%'   : round(auc_tr_10, 4),
        'AUC_val_10%'     : round(auc_va_10, 4),
        'AUC_train_100%'  : round(auc_tr_100, 4),
        'AUC_val_100%'    : round(auc_va_100, 4),
        'Gap_final'       : round(gap, 4),
        'Convergence'     : conv,
        'Diagnostic'      : lc['diagnostic'],
    })

recap_df = pd.DataFrame(recap_rows)
print(recap_df.to_string(index=False))
recap_df.to_csv('../outputs/tables/08_learning_curves_recap.csv', index=False)
print('\n✓ Sauvegardé → outputs/tables/08_learning_curves_recap.csv')

# ── ÉTAPE 5 — Interprétation automatique ─────────────────
print('\n\n' + '=' * 58)
print('INTERPRÉTATIONS DÉTAILLÉES — LEARNING CURVES')
print('=' * 58)

for nom, lc in lc_resultats.items():
    gap   = lc['gap_final']
    auc_v = lc['auc_val']
    auc_t = lc['auc_train']
    conv  = lc['convergence']
    diag  = lc['diagnostic']

    print(f'\n{"="*55}')
    print(f'  MODÈLE : {nom}{"  <- RETENU" if nom == best_nom else ""}')
    print(f'{"="*55}')
    print(f'  AUC validation (100% train)  : {auc_v:.4f}')
    print(f'  AUC entraînement (100% train): {auc_t:.4f}')
    print(f'  Gap (train - validation)     : {gap:.4f}')
    print()

    if gap > 0.20:
        print('  -> OVERFITTING SÉVÈRE détecté')
        print('     Le modèle mémorise les données d\'entraînement')
        print('     mais généralise mal sur de nouvelles données.')
        print('     Solutions possibles :')
        print('       · Régularisation plus forte (C plus petit pour SVM)')
        print('       · Pruning plus agressif (max_depth réduit pour arbre)')
        print('       · Réduire n_estimators ou ajouter min_samples_leaf')

    elif gap > 0.10:
        print('  -> LÉGER OVERFITTING')
        print('     Le modèle généralise correctement mais pourrait')
        print('     être amélioré par une régularisation supplémentaire.')
        print('     Acceptable pour un mémoire de M1.')

    elif auc_v < 0.60:
        print('  -> UNDERFITTING détecté')
        print('     Le modèle est trop simple pour capturer')
        print('     les patterns des données.')
        print('     Solutions : ajouter des features, augmenter la complexité')

    elif conv and auc_v >= 0.75:
        print('  -> BON MODÈLE')
        print('     Les courbes convergent vers un plateau élevé.')
        print('     Le modèle apprend correctement et généralise bien.')
        print('     Ajout de données supplémentaires améliorerait peu les résultats.')

    else:
        print('  -> A SURVEILLER')
        print('     Le comportement du modèle est acceptable mais')
        print('     la convergence n\'est pas totalement stabilisée.')

    print(f'\n  Conclusion : {diag}')

print(f'\n{"="*58}')
print('FIGURES GÉNÉRÉES — LEARNING CURVES')
print(f'{"="*58}')
print('  fig_learning_curves_6modeles.png  (vue d\'ensemble 2x3)')
print('  fig_learning_curves_focus.png     (SVM vs RL, zones colorées)')

ÉTAPE 4 — Tableau récapitulatif des learning curves

         Modèle  AUC_train_10%  AUC_val_10%  AUC_train_100%  AUC_val_100%  Gap_final Convergence Diagnostic
Rég. Logistique         0.8228       0.8025          0.8193        0.8182     0.0010         Oui Bon modèle
  Random Forest         0.9995       0.8015          0.9999        0.9926     0.0073         Oui Bon modèle
 Arbre Décision         0.8603       0.7444          0.8991        0.8939     0.0052         Oui Bon modèle
      SVM (RBF)         0.9448       0.6901          0.9741        0.9693     0.0048         Oui Bon modèle
      KNN (k=5)         0.9384       0.6826          0.9946        0.9756     0.0190         Oui Bon modèle
        XGBoost         0.9988       0.8059          0.9979        0.9884     0.0095         Oui Bon modèle

✓ Sauvegardé → outputs/tables/08_learning_curves_recap.csv


INTERPRÉTATIONS DÉTAILLÉES — LEARNING CURVES

  MODÈLE : Rég. Logistique
  AUC validation (100% train)  : 0.8182
  AUC entraîneme

In [34]:
# ── Fig B2 : Boxplot AUC par fold (stabilité des 6 modèles) ─────────────
print('Fig B2 : Boxplot AUC par fold...')

noms_modeles = list(auc_folds.keys())
data_box = [auc_folds[nom] for nom in noms_modeles]

fig, ax = plt.subplots(figsize=(12, 6))
bp = ax.boxplot(data_box, patch_artist=True, notch=False, vert=True,
                medianprops=dict(color='white', linewidth=2.5))

for i, (patch, nom) in enumerate(zip(bp['boxes'], noms_modeles)):
    col = GREEN if nom == best_nom else COLORS_6[i % len(COLORS_6)]
    patch.set_facecolor(col)
    patch.set_alpha(0.8)

    # Points individuels (strip plot)
    x_jitter = np.random.normal(i+1, 0.07, size=len(data_box[i]))
    ax.scatter(x_jitter, data_box[i], color=col, s=40, zorder=5, alpha=0.9, edgecolors='white', linewidths=0.5)

ax.set_xticks(range(1, len(noms_modeles)+1))
ax.set_xticklabels(noms_modeles, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('AUC-ROC (5 folds)', fontsize=11)
ax.set_title('Fig B2 — Stabilité des modèles : Boxplot AUC par fold (CV-5)\n'
             f'Modèle retenu en vert : {best_nom}', fontsize=12, fontweight='bold')
ax.axhline(0.75, color=GRAY, linestyle='--', linewidth=1, alpha=0.7, label='Seuil acceptable 0.75')
ax.legend(fontsize=9)

for i, nom in enumerate(noms_modeles):
    if nom == best_nom:
        ax.annotate('RETENU', xy=(i+1, np.max(data_box[i])+0.005),
                    ha='center', fontsize=8, color=GREEN, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/figC_boxplot_auc_folds.png', bbox_inches='tight', dpi=130)
plt.show()
print('✓ Fig B2 sauvegardée → figC_boxplot_auc_folds.png')

Fig B2 : Boxplot AUC par fold...
✓ Fig B2 sauvegardée → figC_boxplot_auc_folds.png


In [35]:
# ── Fig B3 : 6 Matrices de confusion côte à côte ────────────────────────
print('Fig B3 : 6 matrices de confusion...')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Fig B3 — Matrices de confusion des 6 modèles ML\n(données de test, n={}  — SMOTE appliqué sur le train uniquement)'.format(len(y_test)),
             fontsize=13, fontweight='bold')

for ax, row in zip(axes.ravel(), resultats):
    nom  = row['Modèle']
    cm_i = row['cm']
    tn_i, fp_i, fn_i, tp_i = cm_i.ravel()
    s_i  = tp_i / (tp_i + fn_i) * 100 if (tp_i + fn_i) > 0 else 0
    sp_i = tn_i / (tn_i + fp_i) * 100 if (tn_i + fp_i) > 0 else 0
    ok_i = (tp_i + tn_i) / len(y_test) * 100

    cm_labels = [
        [f'VN={tn_i:,}', f'FP={fp_i:,}'],
        [f'FN={fn_i:,}', f'VP={tp_i:,}']
    ]
    im = ax.imshow([[tn_i, fp_i], [fn_i, tp_i]], cmap='Blues', interpolation='nearest')
    for ii in range(2):
        for jj in range(2):
            val = cm_i[ii, jj]
            ax.text(jj, ii, cm_labels[ii][jj], ha='center', va='center', fontsize=11,
                    color='white' if val > cm_i.max() / 2 else 'black', fontweight='bold')
    border_col = GREEN if nom == best_nom else GRAY
    for spine in ax.spines.values():
        spine.set_edgecolor(border_col)
        spine.set_linewidth(3 if nom == best_nom else 1)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Prédit : 0', 'Prédit : 1'], fontsize=9)
    ax.set_yticklabels(['Réel : 0', 'Réel : 1'], fontsize=9)
    retenu = ' ← RETENU' if nom == best_nom else ''
    ax.set_title(f'{nom}{retenu}\nSensib: {s_i:.1f}%  Specif: {sp_i:.1f}%  OK: {ok_i:.1f}%',
                 fontsize=9.5, fontweight='bold',
                 color=GREEN if nom == best_nom else 'black')

plt.tight_layout()
plt.savefig('../outputs/figures/figD_matrices_confusion_all.png', bbox_inches='tight', dpi=130)
plt.show()
print('✓ Fig B3 sauvegardée → figD_matrices_confusion_all.png')

Fig B3 : 6 matrices de confusion...
✓ Fig B3 sauvegardée → figD_matrices_confusion_all.png


In [36]:
# ── Fig B4 : Courbes de calibration des 6 modèles ───────────────────────
print('Fig B4 : Courbes de calibration...')

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.8, label='Calibration parfaite', alpha=0.7)

for (nom, color), row in zip(zip(noms_modeles, COLORS_6), resultats):
    yp = row['y_proba_test']
    try:
        frac_i, pred_i = calibration_curve(y_test, yp, n_bins=10, strategy='uniform')
        b_i = brier_scores[nom]
        lw  = 3.0 if nom == best_nom else 1.5
        alp = 1.0 if nom == best_nom else 0.75
        ax.plot(pred_i, frac_i, 'o-', color=color, linewidth=lw, markersize=6,
                label=f'{nom}  (Brier={b_i:.4f})', alpha=alp)
    except Exception:
        pass

ax.set_xlabel('Probabilité prédite moyenne', fontsize=11)
ax.set_ylabel('Fraction de cas positifs observés', fontsize=11)
ax.set_title('Fig B4 — Courbes de calibration des 6 modèles\n'
             '(plus proche de la diagonale = probabilités plus fiables)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../outputs/figures/figE_calibration_curves.png', bbox_inches='tight', dpi=130)
plt.show()
print('✓ Fig B4 sauvegardée → figE_calibration_curves.png')

Fig B4 : Courbes de calibration...
✓ Fig B4 sauvegardée → figE_calibration_curves.png


In [37]:

# ÉTAPE 6E — Analyse SHAP (interprétabilité)
# Fig F1 : Bar plot importance globale  (Régression Logistique)
# Fig F2 : Beeswarm plot
# Fig F3 : Waterfall — une femme bien classée "Ne désire pas"
# + Permutation Importance pour le modèle ML retenu

import shap
print('SHAP — Calcul des valeurs SHAP pour la Régression Logistique...')

# Noms lisibles pour les axes
labels_shap = [LABELS.get(v, v) for v in VARS_MODELE]

# DataFrame X_test avec noms de colonnes lisibles
X_test_df = pd.DataFrame(X_test, columns=labels_shap)

# LinearExplainer est optimal pour les modèles linéaires (rapide et exact)
clf_rl = modeles['Rég. Logistique']
explainer   = shap.LinearExplainer(clf_rl, X_test,
                                    feature_names=labels_shap)
shap_values = explainer(X_test)

print(f'  ✓ SHAP calculé — shape : {shap_values.values.shape}')
print(f'    {len(X_test):,} observations × {len(VARS_MODELE)} variables')

# ── Fig F1 : Bar plot importance globale ─────────────────
mean_abs = np.abs(shap_values.values).mean(axis=0)
shap_imp = pd.DataFrame({'Variable': labels_shap, 'SHAP': mean_abs})
shap_imp = shap_imp.sort_values('SHAP', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(VARS_MODELE) * 0.42)))
colors_s = [NAVY if v >= shap_imp['SHAP'].median() else BLUE
            for v in shap_imp['SHAP']]
bars = ax.barh(shap_imp['Variable'], shap_imp['SHAP'],
               color=colors_s, edgecolor='white', linewidth=1.1)
for b in bars:
    ax.text(b.get_width() + shap_imp['SHAP'].max()*0.01,
            b.get_y() + b.get_height()/2,
            f'{b.get_width():.4f}', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Importance SHAP moyenne (|valeur|)', fontsize=11)
ax.set_title('Fig F1 — Importance des variables (SHAP)\n'
             'Régression Logistique — EDSC Cameroun 2018 (encodage corrigé)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/figF1_shap_importance.png', bbox_inches='tight', dpi=130)
plt.show()
print('✓ Fig F1 sauvegardée → figF1_shap_importance.png')

# ── Fig F2 : Beeswarm plot ───────────────────────────────
plt.figure(figsize=(11, max(6, len(VARS_MODELE) * 0.42)))
shap.plots.beeswarm(shap_values, max_display=len(VARS_MODELE), show=False)
plt.title('Fig F2 — SHAP Beeswarm : distribution des contributions par variable\n'
          'Rouge = valeur élevée · Bleu = valeur faible · Axe X = impact sur la prédiction',
          fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../outputs/figures/figF2_shap_beeswarm.png', bbox_inches='tight', dpi=130)
plt.show()
print('✓ Fig F2 sauvegardée → figF2_shap_beeswarm.png')

# ── Fig F3 : Waterfall — une femme "Ne désire pas" bien classée ──
clf_rl_pred = clf_rl.predict(X_test)
idx_classe0 = np.where((y_test == 0) & (clf_rl_pred == 0))[0]

if len(idx_classe0) > 0:
    idx = idx_classe0[0]
    plt.figure(figsize=(11, 5.5))
    shap.plots.waterfall(shap_values[idx], show=False)
    obs_age  = X_test[idx, VARS_MODELE.index('age')]
    obs_enf  = X_test[idx, VARS_MODELE.index('nb_enfants')]
    plt.title(f'Fig F3 — SHAP Waterfall : femme n°{idx} (prédiction correcte : Ne désire pas)\n'
              f'Âge = {obs_age:.0f} ans · Nb enfants vivants = {obs_enf:.0f}',
              fontsize=11, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig('../outputs/figures/figF3_shap_waterfall.png', bbox_inches='tight', dpi=130)
    plt.show()
    print(f'✓ Fig F3 sauvegardée → figF3_shap_waterfall.png  (obs. n°{idx})')
else:
    print('⚠ Aucune observation bien classée dans la classe 0 — waterfall ignoré')

# ── Résumé SHAP ──────────────────────────────────────────
print('\nRanking SHAP (variables les plus influentes) :')
top_shap = shap_imp.sort_values('SHAP', ascending=False)
for _, row in top_shap.iterrows():
    idx_col   = labels_shap.index(row['Variable'])
    direction = shap_values.values[:, idx_col].mean()
    sens = '→ réduit le désir' if direction < 0 else '→ augmente le désir'
    print(f'  {row["Variable"]:<38} SHAP={row["SHAP"]:.4f}  {sens}')

Background dataset has 2706 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=2706 when initializing the masker.


SHAP — Calcul des valeurs SHAP pour la Régression Logistique...
  ✓ SHAP calculé — shape : (2706, 19)
    2,706 observations × 19 variables
✓ Fig F1 sauvegardée → figF1_shap_importance.png
✓ Fig F2 sauvegardée → figF2_shap_beeswarm.png
✓ Fig F3 sauvegardée → figF3_shap_waterfall.png  (obs. n°12)

Ranking SHAP (variables les plus influentes) :
  Âge                                    SHAP=1.1631  → augmente le désir
  Mariée (vs jamais mariée)              SHAP=0.3120  → augmente le désir
  Nb enfants vivants                     SHAP=0.3004  → augmente le désir
  Résidence rurale (vs urbaine)          SHAP=0.2717  → augmente le désir
  Région septentrionale                  SHAP=0.1610  → augmente le désir
  Musulmane (vs Catholique)              SHAP=0.1354  → augmente le désir
  Protestante (vs Catholique)            SHAP=0.1296  → réduit le désir
  Contraceptif moderne                   SHAP=0.1249  → augmente le désir
  Niveau d'instruction                   SHAP=0.0892  → réduit le

In [38]:
#  Fig F4 : Permutation Importance — modèle ML retenu 
# SHAP LinearExplainer ne s'applique pas à SVM/RF/etc.
# → On utilise Permutation Importance : on permute chaque variable
#   et on mesure la chute d'AUC (variable importante = grande chute)
from sklearn.inspection import permutation_importance

print(f'Permutation Importance — {best_nom}...')

use_sc_best = best_nom in ['SVM (RBF)', 'KNN (k=5)']
Xte_perm    = X_test_sc if use_sc_best else X_test

r_perm = permutation_importance(
    best_clf, Xte_perm, y_test,
    n_repeats=10, random_state=42, scoring='roc_auc', n_jobs=-1
)

perm_df = pd.DataFrame({
    'Variable'   : labels_shap,
    'Chute_AUC'  : r_perm.importances_mean,
    'Ecart_type' : r_perm.importances_std,
}).sort_values('Chute_AUC', ascending=False).reset_index(drop=True)

print(f'\nPermutation Importance — {best_nom} (chute AUC quand variable permutée) :')
print(f'{"Variable":<38} {"Chute AUC":>10} {"Ecart-type":>12}')
print('-' * 62)
for _, row in perm_df.iterrows():
    print(f'{row["Variable"]:<38} {row["Chute_AUC"]:>+10.4f} {row["Ecart_type"]:>12.4f}')

# Figure
perm_plot = perm_df.sort_values('Chute_AUC', ascending=True)
fig, ax = plt.subplots(figsize=(10, max(6, len(VARS_MODELE) * 0.42)))
colors_p = [NAVY if v >= 0 else CORAL for v in perm_plot['Chute_AUC']]
bars = ax.barh(
    perm_plot['Variable'], perm_plot['Chute_AUC'],
    xerr=perm_plot['Ecart_type'], color=colors_p,
    edgecolor='white', linewidth=1.1,
    capsize=4, error_kw={'elinewidth': 1.2, 'ecolor': GRAY}
)
ax.axvline(0, color='gray', linewidth=1.2, linestyle='--', alpha=0.7)
for b, v in zip(bars, perm_plot['Chute_AUC']):
    offset = perm_df['Chute_AUC'].abs().max() * 0.01
    ha = 'left' if v >= 0 else 'right'
    ax.text(b.get_width() + (offset if v >= 0 else -offset),
            b.get_y() + b.get_height()/2,
            f'{v:+.4f}', va='center', fontsize=9, ha=ha)
ax.set_xlabel('Chute d\'AUC-ROC quand la variable est permutée\n'
              '(valeur positive = variable importante ; proche de 0 = peu utile)',
              fontsize=10)
ax.set_title(f'Fig F4 — Permutation Importance — {best_nom} (modèle retenu)\n'
             'EDSC Cameroun 2018 — encodage corrigé',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/figF4_permutation_importance.png', bbox_inches='tight', dpi=130)
plt.show()
print('✓ Fig F4 sauvegardée → figF4_permutation_importance.png')

# Export CSV
perm_df.to_csv('../outputs/tables/08_permutation_importance.csv', index=False)
print('✓ CSV sauvegardé → outputs/tables/08_permutation_importance.csv')

Permutation Importance — SVM (RBF)...

Permutation Importance — SVM (RBF) (chute AUC quand variable permutée) :
Variable                                Chute AUC   Ecart-type
--------------------------------------------------------------
Âge                                       +0.3266       0.0195
Niveau d'instruction                      +0.0321       0.0103
Mariée (vs jamais mariée)                 +0.0286       0.0056
Quintile de richesse                      +0.0256       0.0110
Contraceptif traditionnel                 +0.0220       0.0126
Nb enfants vivants                        +0.0182       0.0095
Contraceptif moderne                      +0.0150       0.0067
Résidence rurale (vs urbaine)             +0.0130       0.0038
Protestante (vs Catholique)               +0.0117       0.0036
Emploi (travail)                          +0.0112       0.0059
Union libre (vs jamais mariée)            +0.0112       0.0061
Musulmane (vs Catholique)                 +0.0085       0.0055
Autres

In [39]:

# ÉTAPE 7 — Tableau de synthèse final

print('ÉTAPE 7 — Tableau récapitulatif final...')

# Calibration diagnostics
def calib_diag(brier):
    if brier < 0.05:
        return 'Excellente'
    elif brier < 0.10:
        return 'Bonne'
    elif brier < 0.15:
        return 'Acceptable'
    else:
        return 'Mauvaise'

# Règles de diagnostic modèle
def model_diag(gap, auc):
    if gap > 0.20:
        return 'Overfitting sévère'
    elif gap > 0.10:
        return 'Léger overfitting'
    elif auc < 0.65:
        return 'Modèle insuffisant'
    else:
        return 'Bon modèle'

synthese_rows = []
for row in resultats:
    nom  = row['Modèle']
    diag = model_diag(row['Gap'], row['AUC_test'])
    cal  = calib_diag(row['Brier'])
    synthese_rows.append({
        'Modèle'      : nom,
        'AUC test'    : row['AUC_test'],
        'AUC CV-5'    : row['AUC_CV5'],
        'Std CV-5'    : row['Std_CV5'],
        'Gap'         : row['Gap'],
        'Brier score' : row['Brier'],
        'Calibration' : cal,
        'Diagnostic'  : diag,
    })

synth_df = pd.DataFrame(synthese_rows).sort_values('AUC CV-5', ascending=False).reset_index(drop=True)
synth_df.to_csv('../outputs/tables/07_synthese_validation_complete.csv', index=False)

print('\n' + '='*90)
print('SYNTHÈSE FINALE — VALIDATION COMPLÈTE')
print('='*90)
print(synth_df.to_string(index=False))
print('\nRègles de diagnostic :')
print('  Gap > 0.20     → Overfitting sévère')
print('  Gap 0.10-0.20  → Léger overfitting')
print('  Gap < 0.10 ET AUC > 0.75 → Bon modèle')
print('  AUC < 0.65     → Modèle insuffisant')
print(f'\n✓ Sauvegardé → outputs/tables/07_synthese_validation_complete.csv')

ÉTAPE 7 — Tableau récapitulatif final...

SYNTHÈSE FINALE — VALIDATION COMPLÈTE
         Modèle  AUC test  AUC CV-5  Std CV-5     Gap  Brier score Calibration         Diagnostic
  Random Forest    0.7757    0.9926    0.0007  0.2241       0.0526       Bonne Overfitting sévère
        XGBoost    0.7961    0.9884    0.0012  0.2013       0.0528       Bonne Overfitting sévère
      KNN (k=5)    0.6374    0.9756    0.0006  0.3580       0.1028  Acceptable Overfitting sévère
      SVM (RBF)    0.7949    0.9693    0.0018  0.1811       0.0742       Bonne  Léger overfitting
 Arbre Décision    0.7443    0.8939    0.0023  0.1570       0.1164  Acceptable  Léger overfitting
Rég. Logistique    0.8198    0.8182    0.0043 -0.0007       0.1622    Mauvaise         Bon modèle

Règles de diagnostic :
  Gap > 0.20     → Overfitting sévère
  Gap 0.10-0.20  → Léger overfitting
  Gap < 0.10 ET AUC > 0.75 → Bon modèle
  AUC < 0.65     → Modèle insuffisant

✓ Sauvegardé → outputs/tables/07_synthese_validation_com

In [40]:
# RÉSUMÉ FINAL
print('=' * 65)
print('ANALYSE CORRIGÉE TERMINÉE — EDSC CAMEROUN 2018')
print('=' * 65)
print(f'\nN = {N:,} femmes · {len(VARS_MODELE)} variables (encodage DHS correct)')
print(f'  Désire     : {N_OUI:,} ({N_OUI/N*100:.1f}%)')
print(f'  Ne désire  : {N_NON:,} ({N_NON/N*100:.1f}%)')
print(f'\nCORRECTIONS APPLIQUÉES PAR RAPPORT AUX NOTEBOOKS PRÉCÉDENTS :')
print(f'  1. v313 contraceptif → 2 dummies (trad / moderne) au lieu de binaire')
print(f'  2. v501 statut matrimonial → 5 dummies au lieu de variable continue 0-5')
print(f'  3. v025 résidence → recodée 0=Urbain(réf)/1=Rural au lieu de 1/2')
print(f'\nMODÈLE LOGISTIQUE :')
print(f'  R² Nagelkerke   : {R2_nag:.4f}  ({R2_nag*100:.1f}%)')
print(f'  AUC-ROC         : {auc_rl:.4f}')
print(f'  AIC             : {logit_model.aic:.2f}')
print(f'  Seuil optimal   : {seuil_opt:.2f}  (intersection sensib/specif)')
print(f'\nMODÈLE ML RETENU : {best_nom}')
print(f'  AUC CV-5 = {best_row["AUC_CV5"]:.4f} | AUC test = {best_row["AUC_test"]:.4f} | Gap = {best_row["Gap"]:.4f}')
print(f'\nFICHIERS GÉNÉRÉS :')
for f in sorted(os.listdir('../outputs/tables/')):
    if any(x in f for x in ['corriges', 'synthese']):
        print(f'   {f}')
for f in sorted(os.listdir('../outputs/figures/')):
    if any(x in f for x in ['figA', 'figB', 'figC', 'figD', 'figE']):
        print(f'   {f}')

ANALYSE CORRIGÉE TERMINÉE — EDSC CAMEROUN 2018

N = 13,527 femmes · 19 variables (encodage DHS correct)
  Désire     : 12,782 (94.5%)
  Ne désire  : 745 (5.5%)

CORRECTIONS APPLIQUÉES PAR RAPPORT AUX NOTEBOOKS PRÉCÉDENTS :
  1. v313 contraceptif → 2 dummies (trad / moderne) au lieu de binaire
  2. v501 statut matrimonial → 5 dummies au lieu de variable continue 0-5
  3. v025 résidence → recodée 0=Urbain(réf)/1=Rural au lieu de 1/2

MODÈLE LOGISTIQUE :
  R² Nagelkerke   : 0.2342  (23.4%)
  AUC-ROC         : 0.8090
  AIC             : 4660.87
  Seuil optimal   : 0.95  (intersection sensib/specif)

MODÈLE ML RETENU : SVM (RBF)
  AUC CV-5 = 0.9693 | AUC test = 0.7949 | Gap = 0.1811

FICHIERS GÉNÉRÉS :
   04_odds_ratio_corriges.csv
   05_vif_corriges.csv
   06_comparaison_modeles_corriges.csv
   07_synthese_validation_complete.csv
   figA_croise_age_residence_quintile.png
   figA_validation_logistique.png
   figB_boxplots_variables_continues.png
   figB_learning_curves.png
   figC_boxplot_a